# Top-K Candidate Search — Source 1 vs. FAISS Index

**What changed vs. the version you gave me, and why:**

1. **The nested Python loop with `mapping.iloc[idx]["entity_id"]` is the single biggest issue here.**
   At `BATCH_SIZE=4096` x `TOP_K=20`, that's ~82k scalar `.iloc` lookups per batch, repeated across every
   batch in every Source 1 file — several orders of magnitude slower than it needs to be. Replaced with
   vectorized numpy indexing.
2. **A real correctness bug, not just a speed one: FAISS returns `-1` for a slot with no neighbor**
   (e.g. an index with fewer than `TOP_K` vectors total). `.iloc[-1]` on a DataFrame means *"last row"*,
   not *"missing"* — so the original code would have silently attached the last entity in the mapping
   file as a bogus candidate every time that happened, for every affected query. Now explicitly filtered
   out before building the result.
3. **Hard GPU dependency, same issue as the index-building script** — `faiss.StandardGpuResources()`
   doesn't exist on `faiss-cpu`. Detects availability and falls back to CPU.
4. **File-glob mismatch with the current Step 2 output** — updated to `part_*_meta.parquet`.
5. **Reads `nprobe`/index kind from the `*_info.json` the Step 3 notebook now saves**, instead of having
   no way to know whether the index it's searching is Flat or IVF.
6. **`SPLIT` config variable** instead of a "change shard path" comment — run this notebook once with
   `SPLIT = "train"` and once with `SPLIT = "test"` instead of hand-editing three paths each time.

The `rank` column is kept as-is — it's exactly the signal the earlier pipeline write-up's "gap between the
top and second candidate" feature needs downstream, so it's worth keeping even though nothing here
consumes it yet.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import faiss
from tqdm.auto import tqdm

In [ ]:
# ======================================================
# CONFIG
# ======================================================

SPLIT = "train"  # "train" or "test" - run this notebook once for each

S1_DIR = Path(f"processed/source1_{SPLIT}_embeddings")
FAISS_DIR = Path("faiss")
OUTPUT_DIR = Path(f"candidates/{SPLIT}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = 20
BATCH_SIZE = 4096

INDEX_PATH = FAISS_DIR / f"faiss_{SPLIT}.index"
MAP_PATH = FAISS_DIR / f"faiss_{SPLIT}_mapping.parquet"
INFO_PATH = FAISS_DIR / f"faiss_{SPLIT}_info.json"

In [ ]:
# ======================================================
# LOAD INDEX (GPU if available, CPU otherwise)
# ======================================================

def detect_gpu():
    has_gpu_api = hasattr(faiss, "StandardGpuResources") and hasattr(faiss, "get_num_gpus")
    if not has_gpu_api:
        return False
    try:
        return faiss.get_num_gpus() > 0
    except Exception:
        return False

USE_GPU = detect_gpu()
print(f"FAISS GPU support: {'available' if USE_GPU else 'not available - searching on CPU'}")

cpu_index = faiss.read_index(str(INDEX_PATH))

with open(INFO_PATH, "r", encoding="utf-8") as f:
    index_info = json.load(f)

if index_info["index_kind"] == "ivf":
    cpu_index.nprobe = index_info["nprobe"]

search_index = cpu_index
if USE_GPU:
    try:
        gpu_res = faiss.StandardGpuResources()
        search_index = faiss.index_cpu_to_gpu(gpu_res, 0, cpu_index)
    except Exception as e:
        print(f"  GPU search unavailable ({e}); searching on CPU.")
        search_index = cpu_index

# nprobe doesn't always survive a cpu->gpu conversion cleanly - set it
# again on whichever index we ended up with, if it's an IVF-type index.
if index_info["index_kind"] == "ivf" and hasattr(search_index, "nprobe"):
    search_index.nprobe = index_info["nprobe"]

mapping = pd.read_parquet(MAP_PATH)
mapping_ids = mapping["entity_id"].to_numpy()

In [ ]:
# ======================================================
# SEARCH
# ======================================================

files = sorted(S1_DIR.glob("part_*_meta.parquet"))
print(f"Found {len(files)} Source 1 ({SPLIT}) parts to search")

for file in files:

    out_file = OUTPUT_DIR / file.name.replace("_meta", "_candidates")

    if out_file.exists():
        print(f"[SKIP] {file.name}")
        continue

    df = pd.read_parquet(file, columns=["entity_id", "embedding"])

    embeddings = np.vstack(df["embedding"].tolist()).astype("float32")
    if np.isnan(embeddings).any():
        raise ValueError(f"NaNs found in embeddings for {file.name} - check the Step 2 output.")

    faiss.normalize_L2(embeddings)  # defensive - Step 2 already normalizes before saving
    entity_ids = np.array(df["entity_id"].tolist())

    all_scores = []
    all_indices = []

    for i in tqdm(range(0, len(embeddings), BATCH_SIZE), desc=file.name):
        batch = embeddings[i : i + BATCH_SIZE]
        scores, indices = search_index.search(batch, TOP_K)
        all_scores.append(scores)
        all_indices.append(indices)

    scores = np.vstack(all_scores)
    indices = np.vstack(all_indices)

    # Vectorized: filter out FAISS's -1 "no neighbor" slots, then build
    # the result with numpy indexing instead of a nested Python loop.
    valid_mask = indices != -1
    row_idx, col_idx = np.nonzero(valid_mask)

    result = pd.DataFrame({
        "entity_id": entity_ids[row_idx],
        "candidate_id": mapping_ids[indices[row_idx, col_idx]],
        "rank": col_idx + 1,
        "bge_cosine": scores[row_idx, col_idx],
    })

    result.to_parquet(out_file, index=False)

(OUTPUT_DIR / "_SUCCESS").touch()
print(f"Top-{TOP_K} candidate generation completed for {SPLIT}.")